In [13]:
# ==================================================
# Imports
# ==================================================

from __future__ import annotations

from typing import Iterable, List, Optional, Dict
import os, re, json, hashlib, subprocess
import argparse

import numpy as np
import pandas as pd
from tqdm import tqdm
from config import AudioParams, TextParams
import torch
import torch_xla.core.xla_model as xm
import config
import torch.nn as nn
import math
from typing import Optional


from safetensors.torch import load_file
from datetime import time, datetime

from transformers import Trainer, TrainingArguments, AutoConfig, PreTrainedModel, PretrainedConfig
import evaluate
import time
import sentencepiece as spm

from torch.utils.data import Dataset
from typing import Dict, List
from pathlib import Path
from dataclasses import dataclass

# audio libraries
import librosa
import soundfile as sf

In [10]:
# ==================================================
# Config
# ==================================================


# --- Paths Configuration (DEV MUST SET THE BASE PATH AS ENV VAR ON THEIR MACHINE) ---
COMMON_VOICE_BASE_DATA_DIR = Path(os.getenv("COMMON_VOICE_BASE_PREPROCESSED_DATA_DIR")).expanduser()
COVOST_TSV_PATH = COMMON_VOICE_BASE_DATA_DIR / "covost_v2.en_de.tsv"
OUTPUT_DIR = COMMON_VOICE_BASE_DATA_DIR
PROCESSED_DATA_DIR = OUTPUT_DIR / "processed"
FEATURES_DIR = PROCESSED_DATA_DIR / "features"


# --- Parameters from data_processor.py by @sygrace---
@dataclass
class AudioParams:
    """Parameters for audio processing."""
    sr_target: int = 16000
    min_dur: float = 0.2
    max_dur: float = 20.0
    snr_db_thresh: float = 5.0
    sil_start_db: int = 20
    sil_stop_db: int = 20
    sil_pad_s: float = 0.05
    n_mels: int = 80
    win_ms: int = 25
    hop_ms: int = 10
    workers: int = 8

@dataclass
class TextParams:
    """Parameters for text processing and filtering."""
    spm_vocab_size: int = 16000
    spm_model_type: str = "unigram"
    max_tok_en: int = 200
    max_tok_de: int = 200
    max_char_en: int = 1200
    max_char_de: int = 1200
    max_len_ratio: float = 3.0
    min_tok: int = 1


@dataclass
class DatasetParams:
    """Parameters for dataset loading and splitting."""
    use_subset: bool = True
    subset_fraction: float = 0.01 # Use 100% of training data, 0.01 = 1% of training data, 0.1 = 10% of training data
    subset_size: int = None  # Another way to do it is to specify exact train subset size
    random_seed: int = 42
    split_method: str = "random"  # "random", "first_n", or "stratified"

    # split train data into train/val if needed
    create_val_split: bool = False
    val_split_ratio: float = 0.1  # 10% for validation

    # For subsets reproducibility
    shuffle_before_split: bool = True


# --- Model Config ---
EMBED_DIM = 256
NUM_HEADS = 8
NUM_ENCODER_LAYERS = 4
NUM_DECODER_LAYERS = 4
D_FF = 1024
DROPOUT = 0.1


# --- Training Config ---
TRAINING_OUTPUT_DIR = Path("./models")  # OUTPUT_DIR / "training_results"


import torch

# TPU/XLA imports
try:
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    HAS_TPU = True
except ImportError:
    HAS_TPU = False
    xm = None

def get_device():
    if HAS_TPU:
        try:
            # Try to get XLA device - this will work if TPU is available
            return xm.xla_device()
        except:
            # Fall back to other devices if XLA device fails
            pass

    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

DEVICE = get_device()
IS_TPU = HAS_TPU and (str(DEVICE).startswith('xla'))


# --- Sample Hyperparameters (Just initials, not all are used) ---
BATCH_SIZE = 16
NUM_TRAIN_EPOCHS = 5
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
GRADIENT_ACCUMULATION_STEPS = 2
WARMUP_STEPS = 1000
FP16 = True if DEVICE == "cuda" else False
EVAL_STRATEGY = "steps" # or "epoch" but must adjust save strategy
EVAL_STEPS = 1000
SAVE_STEPS = 1000
LOGGING_STEPS = 200
SAVE_TOTAL_LIMIT = 2
LOAD_BEST_MODEL_AT_END = True
METRIC_FOR_BEST_MODEL = "bleu"
GREATER_IS_BETTER = True


# --- Dataset Config ---
DATASET_PARAMS = DatasetParams()

# Quick access variables for backward compatibility
USE_SUBSET = DATASET_PARAMS.use_subset
SUBSET_FRACTION = DATASET_PARAMS.subset_fraction
SUBSET_SIZE = DATASET_PARAMS.subset_size
RANDOM_SEED = DATASET_PARAMS.random_seed
SPLIT_METHOD = DATASET_PARAMS.split_method



In [3]:
# ====================
# Data Manager for Audio-Text Preprocessing Pipeline from data_processor.py
# ====================

# Choose regex library
try:
    import regex as re2

    _RE = re2
except ImportError:
    _RE = re

try:
    import sentencepiece as spm

    HAS_SPM = True
except ImportError:
    HAS_SPM = False

import warnings

# warnings.filterwarnings("ignore", message="PySoundFile failed. Trying audioread", category=UserWarning)
# warnings.filterwarnings("ignore", category=FutureWarning, module="librosa")


# =====================
# Paths Class
# =====================
@dataclass
class Paths:
    base: Path
    clips: Path
    covost_tsv: Path
    out_dir: Path

    @classmethod
    def make(cls, base: str | Path, covost: str | Path, out: Optional[str | Path] = None) -> "Paths":
        base = Path(base)
        return cls(
            base=base,
            clips=base / "clips",
            covost_tsv=Path(covost),
            out_dir=Path(out) if out else base,
        )


def _ensure(p: Path):
    p.mkdir(parents=True, exist_ok=True)


CV_SPLITS_DEFAULT = ["validated.tsv", "train.tsv", "dev.tsv", "test.tsv"]


# =====================
# Audio Processor
# =====================
class AudioProcessor:
    def __init__(self, paths: Paths, a: AudioParams, splits: Iterable[str] = CV_SPLITS_DEFAULT):
        self.P = paths
        self.A = a
        self.splits = list(splits)
        # Derived paths using the central output directory
        self.artifacts_dir = self.P.out_dir
        self.metrics_csv = self.artifacts_dir / "merged_en_de_metrics.csv"
        self.merged_csv = self.artifacts_dir / "merged_en_de.csv"
        self.filtered_csv = self.artifacts_dir / "merged_en_de_filtered.csv"
        self.wav_manifest_csv = self.artifacts_dir / "manifest_wav.csv"
        self.seed_manifest_csv = self.artifacts_dir / "manifest_seed.csv"
        self.manifest_features_csv = self.artifacts_dir / "manifest_features.csv"
        self.cmvn_json = self.artifacts_dir / "cmvn_train.json"
        self.wav_dir = self.artifacts_dir / "wav"
        self.features_dir = self.artifacts_dir / "features" / "logmel"
        _ensure(self.artifacts_dir)
        _ensure(self.features_dir)
        _ensure(self.wav_dir)

    # ---------- 1. Load & Link ----------
    def load_and_link(self) -> pd.DataFrame:
        total_possible = 0
        merged_dfs: List[pd.DataFrame] = []

        # Report CV split sizes
        for file in self.splits:
            path = self.P.base / file
            try:
                df = pd.read_csv(path, sep="\t", dtype={"accent": str})
                print(f"{file:<20}: {len(df):>8} rows")
                total_possible += len(df)
            except Exception as e:
                print(f"{file:<20}: Error - {e}")
        print()

        # Load CoVoST translations
        covost_df = pd.read_csv(self.P.covost_tsv, sep="\t")
        assert "path" in covost_df.columns, "CoVoST must contain 'path'"

        # Detect translation column
        cands = [c for c in covost_df.columns if
                 c.lower().startswith("translation") or c.endswith("_de") or c.lower() == "de"]
        translation_col = cands[0] if cands else "translation"
        if translation_col not in covost_df.columns:
            raise ValueError("Could not find translation column in CoVoST TSV.")

        # Merge each split with CoVoST
        for file_name in self.splits:
            path = self.P.base / file_name
            if not path.exists():
                continue
            try:
                cv_df = pd.read_csv(path, sep="\t", dtype={"accent": str})
                matched = pd.merge(cv_df, covost_df, on="path")
                print(
                    f"{file_name:<20}: matched {len(matched):>6} / {len(cv_df):>6} rows ({100 * len(matched) / max(len(cv_df), 1):.2f}%)")
                # Construct audio_path + select columns
                matched["audio_path"] = matched["path"].apply(lambda q: str((self.P.clips / q).resolve()))
                en_col = "sentence" if "sentence" in matched.columns else (
                    "text" if "text" in matched.columns else "en_text")
                matched = matched[["audio_path", en_col, translation_col]].rename(
                    columns={en_col: "en_text", translation_col: "de_text"})
                merged_dfs.append(matched)
            except Exception as e:
                print(f"{file_name:<20}: merge error - {e}")

        merged_df = pd.concat(merged_dfs, ignore_index=True) if merged_dfs else pd.DataFrame(
            columns=["audio_path", "en_text", "de_text"])
        merged_df.to_csv(self.merged_csv, index=False)
        print(f"💾 Total matched samples written to: {self.merged_csv}\n")
        return merged_df

    # ---------- 2. EDA & Filtering (Duration, SNR) ----------
    @staticmethod
    def safe_duration(path: str) -> Optional[float]:
        try:
            with sf.SoundFile(path) as f:
                return float(len(f) / f.samplerate)
        except Exception:
            return None

    @staticmethod
    def estimate_snr_librosa(filename: str, sr: int, top_db: int = 20) -> Optional[float]:
        try:
            y, fs = librosa.load(filename, sr=sr, mono=True)
            if y is None or y.size == 0: return None
            total_e = float(np.mean(y ** 2) + 1e-9)
            intervals = librosa.effects.split(y, top_db=top_db)
            signal_e = float(
                np.mean(np.concatenate([y[s:e] for s, e in intervals]) ** 2) + 1e-9) if intervals.size > 0 else 0.0
            noise_e = max(total_e - signal_e, 1e-9)
            return 10.0 * np.log10(signal_e / noise_e)
        except Exception:
            return None

    def compute_metrics_duration_snr(self, merged_csv: Optional[Path] = None) -> pd.DataFrame:
        df = pd.read_csv(merged_csv or self.merged_csv)
        df["duration_sec"] = [self.safe_duration(p) for p in tqdm(df["audio_path"], desc="Compute duration")]
        df["snr_db"] = [self.estimate_snr_librosa(p, self.A.sr_target) for p in
                        tqdm(df["audio_path"], desc="Estimate SNR")]
        df.to_csv(self.metrics_csv, index=False)
        print(f"💾 Saved metrics → {self.metrics_csv}\n")
        return df

    def filter_by_duration_snr(self, metrics_csv: Optional[Path] = None) -> pd.DataFrame:
        df = pd.read_csv(metrics_csv or self.metrics_csv)
        mask = (
                df["duration_sec"].between(self.A.min_dur, self.A.max_dur) &
                (df["snr_db"] >= self.A.snr_db_thresh)
        )
        filtered = df[mask].dropna(subset=["duration_sec", "snr_db"]).copy()
        filtered.to_csv(self.filtered_csv, index=False)
        print(f"Filtered {len(filtered)} / {len(df)} rows → {self.filtered_csv.name}")
        return filtered

    # ---------- 3. Speaker‑Disjoint Split ----------
    def attach_speaker_and_split(self, filtered_csv: Optional[Path] = None, split_ratio=(98, 1, 1)) -> pd.DataFrame:
        fdf = pd.read_csv(filtered_csv or self.filtered_csv)
        client_map = {}
        for file in self.splits:
            p = self.P.base / file
            if p.exists():
                t = pd.read_csv(p, sep="\t", usecols=["path", "client_id"])
                client_map.update(dict(zip(t.path, t.client_id)))
        fdf["path"] = fdf["audio_path"].map(lambda q: os.path.basename(str(q)))
        fdf["client_id"] = fdf["path"].map(client_map).fillna("unknown")
        fdf["split"] = fdf["client_id"].map(lambda cid: self.assign_split_by_speaker(cid, split_ratio))
        fdf.to_csv(self.seed_manifest_csv, index=False)
        print("💾 Saved speaker-disjoint manifest → manifest_seed.csv\n")
        return fdf

    @staticmethod
    def assign_split_by_speaker(client_id: str, ratio=(98, 1, 1)) -> str:
        bucket = int(hashlib.md5(client_id.encode("utf-8")).hexdigest(), 16) % 100
        train_r, dev_r, _ = ratio
        if bucket < train_r:
            return "train"
        elif bucket < train_r + dev_r:
            return "dev"
        else:
            return "test"

    # ---------- 4. Audio Standardization (FFmpeg) ----------
    def standardize_all_to_wav(self, seed_csv: Optional[Path] = None) -> pd.DataFrame:
        from concurrent.futures import ThreadPoolExecutor, as_completed
        seed = pd.read_csv(seed_csv or self.seed_manifest_csv)
        paths = seed["audio_path"].tolist()
        results = [""] * len(paths)
        with ThreadPoolExecutor(max_workers=self.A.workers) as ex:
            futs = {ex.submit(self._ffmpeg_standardize, p): i for i, p in enumerate(paths)}
            for fut in tqdm(as_completed(futs), total=len(futs), desc="Standardizing WAVs"):
                results[futs[fut]] = fut.result()
        wav_manifest = seed.copy()
        wav_manifest["audio_wav"] = [r.as_posix() if r else "" for r in results]
        wav_manifest.to_csv(self.wav_manifest_csv, index=False)
        print("💾 Saved WAV manifest → manifest_wav.csv\n")
        return wav_manifest

    def _ffmpeg_standardize(self, src_path: str) -> Optional[Path]:
        src, out_wav = Path(src_path), (self.wav_dir / Path(src_path).name).with_suffix(".wav")
        af = [f"silenceremove=start_periods=1:start_threshold={self.A.sil_start_db}dB",
              f"stop_periods=1:stop_threshold={self.A.sil_stop_db}dB"]
        cmd = ["ffmpeg", "-nostdin", "-y", "-i", str(src), "-ac", "1", "-ar", str(self.A.sr_target), "-sample_fmt",
               "s16", "-af", ",".join(af), str(out_wav)]
        try:
            subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            return out_wav
        except subprocess.CalledProcessError:
            return None

    # ---------- 5. Feature Extraction + Train-only CMVN ----------
    def extract_features_and_cmvn(self, wav_manifest_csv: Optional[Path] = None) -> pd.DataFrame:
        dfm = pd.read_csv(wav_manifest_csv or self.wav_manifest_csv)
        out_paths, shapes = [], []
        for p in tqdm(dfm["audio_wav"], desc="Extract log-mel feats"):
            F = self._logmel_from_wav(Path(p)) if p and isinstance(p, str) else None
            if F is None:
                out_paths.append("")
                shapes.append("")
                continue
            out_p = self.features_dir / (Path(p).stem + ".npy")
            np.save(out_p, F)
            out_paths.append(out_p.as_posix())
            shapes.append(f"{F.shape[0]}x{F.shape[1]}")
        dfm["feat_npy"], dfm["feat_shape"] = out_paths, shapes
        # CMVN
        train_feats = dfm.loc[dfm["split"] == "train", "feat_npy"].dropna()
        sum_feat, sum_sq, count = None, None, 0
        for p in tqdm(train_feats, desc="CMVN accumulate"):
            F = np.load(p)
            sum_feat = F.sum(axis=1) if sum_feat is None else sum_feat + F.sum(axis=1)
            sum_sq = (F ** 2).sum(axis=1) if sum_sq is None else sum_sq + (F ** 2).sum(axis=1)
            count += F.shape[1]
        mean = (sum_feat / count)
        var = (sum_sq / count - mean ** 2)
        std = np.sqrt(np.maximum(var, 1e-8))
        cmvn = {"mean": mean.tolist(), "std": std.tolist()}
        with open(self.cmvn_json, "w") as f:
            json.dump(cmvn, f)
        for p in tqdm(dfm["feat_npy"].dropna(), desc="Apply CMVN"): np.save(p,
                                                                            (np.load(p) - mean[:, None]) / std[:, None])
        dfm.to_csv(self.manifest_features_csv, index=False)
        print("💾 Saved CMVN stats and final feature manifest.\n")
        return dfm

    def _logmel_from_wav(self, path_wav: Path) -> Optional[np.ndarray]:
        try:
            y, sr = sf.read(str(path_wav))
            n_fft, hop = int(sr * self.A.win_ms / 1000), int(sr * self.A.hop_ms / 1000)
            S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=self.A.n_mels, n_fft=n_fft, hop_length=hop)
            return np.log(np.maximum(S, 1e-10)).astype(np.float32)
        except Exception:
            return None


# =====================
# Text Processor
# =====================
class TextProcessor:
    def __init__(self, paths: Paths, t: TextParams):
        self.P = paths
        self.T = t
        self.artifacts_dir = self.P.out_dir
        self.text_out = self.artifacts_dir / "text_outputs"
        self.vocab_dir = self.text_out / "spm"
        self.cleaned_dir = self.text_out / "cleaned"
        self.encoded_dir = self.text_out / "encoded"
        for d in [self.text_out, self.vocab_dir, self.cleaned_dir, self.encoded_dir]: _ensure(d)
        self.model_prefix = (self.vocab_dir / f"spm_shared_{self.T.spm_model_type}_{self.T.spm_vocab_size}").as_posix()
        self.corpus_txt = self.vocab_dir / "spm_corpus.txt"


    def uclean(self, s: str) -> str:
        import html, unicodedata
        s = html.unescape(str(s if s is not None else ""))
        s = unicodedata.normalize("NFKC", s)
        s = _RE.compile(r"\[(noise|music|laughter)\]", _RE.IGNORECASE).sub(" ", s)
        return _RE.compile(r"\s+").sub(" ", s).strip()

    def clean_en(self, s: str) -> str:
        return self.uclean(s).lower()

    def clean_de(self, s: str) -> str:
        o = self.uclean(s)
        # optional: fix German quotes
        o = o.replace("\u201e", '"').replace("\u201c", '"').replace("\u201f", '"')
        return o

    def load_merge_pairs(self, audio_seed_csv: Optional[Path] = None) -> pd.DataFrame:
        # Use manifest_seed.csv from audio pipeline to ensure identical rows/order
        seed_csv = audio_seed_csv or (self.P.base / "manifest_seed.csv")
        if not seed_csv.exists():
            raise FileNotFoundError("Run AudioProcessor.attach_speaker_and_split() first to create manifest_seed.csv")
        seed = pd.read_csv(seed_csv)
        # Keep only columns we need
        return seed[["audio_path","en_text","de_text","split"]].copy()

    def train_shared_spm(self, merged_seed_csv: Path) -> None:
        if not HAS_SPM: return
        df = pd.read_csv(merged_seed_csv)
        with open(self.corpus_txt, "w", encoding="utf-8") as f:
            for _, r in tqdm(df[df["split"] == "train"].iterrows(), total=len(df[df["split"] == "train"]),
                             desc="Build SPM corpus"):
                f.write("<en> " + self.uclean(r["en_text"]).lower() + "\n")
                f.write("<de> " + self.uclean(r["de_text"]) + "\n")
        spm.SentencePieceTrainer.Train(
            f"--input={self.corpus_txt} --model_prefix={self.model_prefix} "
            f"--vocab_size={self.T.spm_vocab_size} --model_type={self.T.spm_model_type} "
            f"--user_defined_symbols=<en>,<de>"
        )
        print("Trained SPM:", self.model_prefix + ".model\n")

    def tokenize_and_filter(self, merged_seed_csv: Path) -> None:
        df = pd.read_csv(merged_seed_csv)
        sp = spm.SentencePieceProcessor(model_file=self.model_prefix + ".model") if HAS_SPM else None
        for split, sdf in df.groupby("split"):
            rows = []
            for _, r in tqdm(sdf.iterrows(), total=len(sdf), desc=f"Clean/filter {split}"):
                en_c, de_c = self.uclean(r["en_text"]).lower(), self.uclean(r["de_text"])
                if not en_c or not de_c: continue
                en_tok, de_tok = len(sp.encode(en_c)) if sp else len(en_c.split()), len(sp.encode(de_c)) if sp else len(
                    de_c.split())
                ratio = max(en_tok / max(de_tok, 1), de_tok / max(en_tok, 1))
                if (self.T.min_tok <= en_tok <= self.T.max_tok_en and
                        self.T.min_tok <= de_tok <= self.T.max_tok_de and
                        ratio <= self.T.max_len_ratio):
                    rows.append({"audio_path": r["audio_path"], "en_text": r["en_text"], "de_text": r["de_text"]})
            out_p = self.cleaned_dir / f"{split}_clean.tsv"
            pd.DataFrame(rows).to_csv(out_p, sep="\t", index=False)
            print(f"💾 Saved cleaned ({len(rows)} rows) → {out_p.name}\n")

    def _sp(self):
        sp = spm.SentencePieceProcessor()
        sp.load(self.model_prefix + ".model")
        return sp

    def encode_splits(self, df_in: Optional[pd.DataFrame] = None) -> Dict[str, Path]:
        if not HAS_SPM:
            print("[WARN] sentencepiece not installed, skipping encoding")
            return {}
        sp = self._sp()
        df = df_in if df_in is not None else self.load_merge_pairs()
        out: Dict[str, Path] = {}
        for split, sdf in df.groupby("split"):
            rows = []
            for _, r in tqdm(sdf.iterrows(), total=len(sdf), desc=f"Encode {split}"):
                en_c = self.clean_en(r["en_text"])
                de_c = self.clean_de(r["de_text"])
                src_ids = sp.encode("<en> " + en_c, out_type=int)
                tgt_ids = sp.encode("<de> " + de_c, out_type=int)
                rows.append({
                    "audio_path": r["audio_path"],
                    "en_clean": en_c,
                    "de_clean": de_c,
                    "src_ids": " ".join(map(str, src_ids)),
                    "tgt_ids": " ".join(map(str, tgt_ids)),
                    "src_len_tok": len(src_ids),
                    "tgt_len_tok": len(tgt_ids),
                })
            out_df = pd.DataFrame(rows)
            out_p = self.encoded_dir / f"{split}.csv"
            out_df.to_csv(out_p, index=False)
            out[str(split)] = out_p
            print(f"💾 Saved encoded → {out_p}\n")
        return out


def run_preprocessing(base_dir, covost_tsv, output_dir):
    """Sort of main function to run the local offline preprocessing pipeline.
       No need to run when you already have preprocessed data.
       Run in part by commenting out steps if needed.
    """
    paths = Paths.make(base=base_dir, covost=covost_tsv, out=output_dir)
    print(f"Using base data dir: {paths}")
    audio_params = AudioParams()
    text_params = TextParams()

    print("========== AUDIO PREPROCESSING ==========")
    ap = AudioProcessor(paths, audio_params)
    ap.load_and_link()
    ap.compute_metrics_duration_snr()
    ap.filter_by_duration_snr()
    ap.attach_speaker_and_split()
    ap.standardize_all_to_wav()
    ap.extract_features_and_cmvn()
    print("--- Audio Preprocessing Complete ---")

    print("\n========== TEXT PREPROCESSING ==========")
    tp = TextProcessor(paths, text_params)
    seed_manifest = tp.load_merge_pairs()  # ap.seed_manifest_csv
    # print(f"Using seed manifest: {seed_manifest}")
    tp.train_shared_spm(ap.seed_manifest_csv)
    tp.tokenize_and_filter(ap.seed_manifest_csv)
    tp.encode_splits(seed_manifest)
    print("--- Text Preprocessing Complete ---")


In [4]:
# ==================================================
# Data Loader
# ==================================================




class PreprocessedDataset(Dataset):
    """
    PyTorch-like Dataloader to load the local preprocessed data from
    data_processor.py | data_manger.py.
    """

    def __init__(self, split: str, base_output_dir: Path, subset_params=None):
        """
        Args:
            split (str): The dataset split to load ('train', 'dev', or 'test').
            base_output_dir (Path): The main output directory where artifacts are stored.
        """
        self.split = split
        self.base_dir = base_output_dir
        self.features_dir = FEATURES_DIR
        self.logmels_dir = self.features_dir / "logmel"

        # Load the feature manifest to get paths to .npy files
        features_manifest_path = self.base_dir / "manifest_features.csv"
        # Load the encoded text manifest to get token ID sequences
        encoded_text_path = self.base_dir / "text_outputs" / "encoded" / f"{split}.csv"

        if not features_manifest_path.exists() or not encoded_text_path.exists():
            raise FileNotFoundError(
                f"Manifests not found. Please run the preprocessing step first. "
                f"Checked for: {features_manifest_path} and {encoded_text_path}"
            )

        # Merge the two manifests on 'audio_path' to align features and labels
        df_features = pd.read_csv(features_manifest_path)
        df_encoded = pd.read_csv(encoded_text_path)

        # Filter features manifest for the correct split
        df_features_split = df_features[df_features['split'] == self.split].copy()

        self.manifest = pd.merge(
            df_features_split, df_encoded, on="audio_path", how="inner"
        ).dropna(subset=['feat_npy', 'tgt_ids'])

        # Use subset of train data if specified (to save time during development)
        if subset_params and subset_params.use_subset and split == "train":
            self._apply_subset(subset_params)


    def __len__(self) -> int:
        return len(self.manifest)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        """
        Loads a single data sample from disk.
        """
        sample = self.manifest.iloc[idx]

        # Load the log-mel feature matrix from the .npy file and get file name
        sample_feature_file = sample["feat_npy"].split('/')[-1]
        print(f"Loading features from: {sample_feature_file}")

        feature_path = self.logmels_dir / sample_feature_file

        features = np.load(feature_path).T  # Transpose to (Time, Freq)

        # The target IDs are stored as a space-separated string
        target_ids = list(map(int, sample["tgt_ids"].split()))

        return {
            "input_features": torch.from_numpy(features).float(),
            "labels": torch.tensor(target_ids, dtype=torch.long),
        }

    def _apply_subset(self, params):
        """Apply subsetting to the dataset."""
        original_size = len(self.manifest)
        print(f"Applying subset: original size {original_size}")

        if params.subset_size:
            print(f"Using fixed subset size: {params.subset_size}")
            n_samples = min(params.subset_size, original_size)
            print(f"Adjusted subset size to {n_samples} based on available data.")
        else:
            n_samples = int(original_size * params.subset_fraction)
            print(f"Using subset fraction: {params.subset_fraction}, resulting in {n_samples} samples.")

        if params.split_method == "random":
            self.manifest = self.manifest.sample(n=n_samples, random_state=params.random_seed)
        elif params.split_method == "first_n":
            self.manifest = self.manifest.head(n_samples)

        # Reset index
        self.manifest = self.manifest.reset_index(drop=True)

        print(f"Using subset of training data: {len(self.manifest)} samples out of {original_size}")


@dataclass
class PaddingDataCollator:
    """
    Pads features and labels to the same length in batches
    """

    def __call__(self, features: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # Pad  features - log-mels
        batch = {}
        inputs_padded = torch.nn.utils.rnn.pad_sequence(
            [f["input_features"] for f in input_features], batch_first=True, padding_value=0.0
        )
        batch["input_features"] = inputs_padded

        # Pad token IDs
        tokens_padded = torch.nn.utils.rnn.pad_sequence(
            [f["input_ids"] for f in label_features], batch_first=True, padding_value=1
        )
        batch["labels"] = tokens_padded

        return batch

In [31]:
# ==================================================
# Model - Encoder-Decoder Transformer
# ==================================================

class PositionalFF(nn.Module):
    def __init__(self, embed_dim, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, d_ff)
        self.fc2 = nn.Linear(d_ff, embed_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x): return self.fc2(self.dropout(self.relu(self.fc1(x))))


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, dropout: float):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim, self.num_heads, self.head_dim = embed_dim, num_heads, embed_dim // num_heads
        self.q_proj, self.k_proj, self.v_proj, self.out_proj = (nn.Linear(embed_dim, embed_dim) for _ in range(4))
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        bs = query.shape[0]
        query, key, value = (proj(x).view(bs, -1, self.num_heads, self.head_dim).transpose(1, 2) for proj, x in
                             [(self.q_proj, query), (self.k_proj, key), (self.v_proj, value)])
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None: scores = scores.masked_fill(mask == 0, -1e9)
        attn = self.dropout(torch.softmax(scores, dim=-1))
        ctx = torch.matmul(attn, value).transpose(1, 2).contiguous().view(bs, -1, self.embed_dim)
        return self.out_proj(ctx)


class EncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.self_attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.feed_forward = PositionalFF(embed_dim, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, mask):
        norm_src = self.norm1(src)
        src = src + self.dropout(self.self_attn(norm_src, norm_src, norm_src, mask))
        norm_src = self.norm2(src)
        src = src + self.dropout(self.feed_forward(norm_src))
        return src


class DecoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.self_attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.cross_attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.norm3 = nn.LayerNorm(embed_dim)
        self.feed_forward = PositionalFF(embed_dim, d_ff, dropout)
        self.dropout =  nn.Dropout(dropout)

    def forward(self, tgt, mem, tgt_mask, mem_mask):
        norm_tgt = self.norm1(tgt)
        tgt = tgt + self.dropout(self.self_attn(norm_tgt, norm_tgt, norm_tgt, tgt_mask))
        norm_tgt = self.norm2(tgt)
        tgt = tgt + self.dropout(self.cross_attn(norm_tgt, mem, mem, mem_mask))
        norm_tgt = self.norm3(tgt)
        tgt = tgt + self.dropout(self.feed_forward(norm_tgt))
        return tgt


class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, dropout, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(1, max_len, embed_dim)
        pos, div = torch.arange(max_len).unsqueeze(1), torch.exp(
            torch.arange(0, embed_dim, 2) * (-math.log(10000.0) / embed_dim))
        pe[0, :, 0::2], pe[0, :, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class SpeechToTextTranslationConfig(PretrainedConfig):
    model_type = "speech_to_text_translation"

    def __init__(
        self,
        num_attn_heads=8,
        tgt_vocab_size=16000,
        num_encoder_layers=4,
        num_decoder_layers=4,
        embed_dim=256,
        d_ff=1024,
        dropout=0.1,
        input_feat_dim=80,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.num_attn_heads = num_attn_heads
        self.tgt_vocab_size = tgt_vocab_size
        self.num_encoder_layers = num_encoder_layers
        self.num_decoder_layers = num_decoder_layers
        self.embed_dim = embed_dim
        self.d_ff = d_ff
        self.dropout = dropout
        self.input_feat_dim = input_feat_dim

"""class SpeechToTextTranslationConfig(PretrainedConfig):
    model_type = "speech_to_text_translation"

    def __init__(
        self,
        num_attn_heads=8,
        tgt_vocab_size=16000,
        num_encoder_layers=4,
        num_decoder_layers=4,
        embed_dim=256,
        d_ff=1024,
        dropout=0.1,
        input_feat_dim=80,
        pad_token_id=1,
        bos_token_id=0,
        eos_token_id=2,
        **kwargs
    ):
        super().__init__(
            pad_token_id=pad_token_id,
            bos_token_id=bos_token_id,
            eos_token_id=eos_token_id,
            **kwargs
        )
        self.num_attn_heads = num_attn_heads
        self.tgt_vocab_size = tgt_vocab_size
        self.num_encoder_layers = num_encoder_layers
        self.num_decoder_layers = num_decoder_layers
        self.embed_dim = embed_dim
        self.d_ff = d_ff
        self.dropout = dropout
        self.input_feat_dim = input_feat_dim"""


"""class SpeechToTextTranslationModel(PreTrainedModel):
    config_class = SpeechToTextTranslationConfig

    def __init__(self, config):
        super().__init__(config)
        self.config = config

        self.feature_projection = nn.Linear(config.input_feat_dim, config.embed_dim)
        self.pos_encoder = PositionalEncoding(config.embed_dim, config.dropout)
        self.tgt_embedding = nn.Embedding(config.tgt_vocab_size, config.embed_dim)

        self.encoder_stack = nn.ModuleList([
            EncoderLayer(config.embed_dim, config.num_attn_heads, config.d_ff, config.dropout)
            for _ in range(config.num_encoder_layers)
        ])
        self.decoder_stack = nn.ModuleList([
            DecoderLayer(config.embed_dim, config.num_attn_heads, config.d_ff, config.dropout)
            for _ in range(config.num_decoder_layers)
        ])

        self.generator = nn.Linear(config.embed_dim, config.tgt_vocab_size)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=config.pad_token_id)

        # Initialize weights
        self.post_init()

    def _make_pad_lookahead_target_mask(self, target):
        device = target.device
        target_len = target.shape[1]
        target_pad_mask = (target != self.config.pad_token_id).unsqueeze(1).unsqueeze(2)
        target_lookahead_mask = torch.triu(
            torch.ones((target_len, target_len), device=device, dtype=torch.bool),
            diagonal=1
        )
        return target_pad_mask & ~target_lookahead_mask

    def forward(self, input_features, labels=None, **kwargs):
        # Ensure tensors are on the correct device
        src = self.feature_projection(input_features) * math.sqrt(self.config.embed_dim)
        src = self.pos_encoder(src)

        memory = src
        for layer in self.encoder_stack:
            memory = layer(memory, None)

        if labels is not None:
            tgt_mask = self._make_pad_lookahead_target_mask(labels)
            tgt_emb = self.tgt_embedding(labels) * math.sqrt(self.config.embed_dim)
            tgt_emb = self.pos_encoder(tgt_emb)
            dec_output = tgt_emb
            for layer in self.decoder_stack:
                dec_output = layer(dec_output, memory, tgt_mask, None)
            logits = self.generator(dec_output)
            loss = self.loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
            return {"logits": logits, "loss": loss}

        return {"encoder_out": memory}

    def get_input_embeddings(self):
        return self.tgt_embedding

    def set_input_embeddings(self, value):
        self.tgt_embedding = value"""


class SpeechToTextTranslationModel(nn.Module):
    def __init__(self, pconfig):
        super().__init__()
        self.config = pconfig

        self.feature_projection = nn.Linear(pconfig.input_feat_dim, pconfig.embed_dim)
        self.pos_encoder = PositionalEncoding(pconfig.embed_dim, pconfig.dropout)
        self.tgt_embedding = nn.Embedding(pconfig.tgt_vocab_size, pconfig.embed_dim)

        self.encoder_stack = nn.ModuleList(
            [EncoderLayer(pconfig.embed_dim, pconfig.num_attn_heads, pconfig.d_ff, pconfig.dropout) for _ in range(pconfig.num_encoder_layers)])
        self.decoder_stack = nn.ModuleList(
            [DecoderLayer(pconfig.embed_dim, pconfig.num_attn_heads, pconfig.d_ff, pconfig.dropout) for _ in range(pconfig.num_decoder_layers)])

        self.generator = nn.Linear(pconfig.embed_dim, pconfig.tgt_vocab_size)
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=1)  # Assume pad token index is 1

    def _make_pad_lookahead_target_mask(self, target, device):
        target_len = target.shape[1]
        target_pad_mask = (target != 1).unsqueeze(1).unsqueeze(2)
        target_lookahead_mask = torch.triu(torch.ones((target_len, target_len), device=device), diagonal=1).bool()
        return target_pad_mask & ~target_lookahead_mask

    def forward(self, input_features, labels=None, **kwargs):
        src = self.feature_projection(input_features) * math.sqrt(self.config.embed_dim)
        src = self.pos_encoder(src)

        memory = src
        for layer in self.encoder_stack:
            memory = layer(memory, None)

        if labels is not None:
            tgt_mask = self._make_pad_lookahead_target_mask(labels, labels.device)
            tgt_emb = self.tgt_embedding(labels) * math.sqrt(self.config.embed_dim)
            tgt_emb = self.pos_encoder(tgt_emb)
            dec_output = tgt_emb
            for layer in self.decoder_stack:
                dec_output = layer(dec_output, memory, tgt_mask, None)
            logits = self.generator(dec_output)
            loss = self.loss_fn(logits.view(-1, logits.shape[-1]), labels.view(-1))
            return {"logits": logits, "loss": loss}

        return {"encoder_out": memory}


In [28]:
# ==================================================
# Training Manager
# ==================================================



class TrainingManager:
    def __init__(self):
        self.train_dataset = PreprocessedDataset(
            split="train",
            base_output_dir=OUTPUT_DIR,
            subset_params=DATASET_PARAMS
        )
        self.eval_dataset = PreprocessedDataset(split="dev", base_output_dir=OUTPUT_DIR)

        # Load the preprocessed SentencePiece model
        spm_path = OUTPUT_DIR / "text_outputs" / "spm" / f"spm_shared_{TextParams.spm_model_type}_{TextParams.spm_vocab_size}.model"
        self.sentence_piece = spm.SentencePieceProcessor(model_file=str(spm_path))

        config = SpeechToTextTranslationConfig(
            num_encoder_layers=NUM_ENCODER_LAYERS,
            num_decoder_layers=NUM_DECODER_LAYERS,
            embed_dim=EMBED_DIM,
            num_attn_heads=NUM_HEADS,
            tgt_vocab_size=self.sentence_piece.get_piece_size(),
            d_ff=D_FF,
            dropout=DROPOUT,
            input_feat_dim=AudioParams.n_mels
        )

        self.model = SpeechToTextTranslationModel(config)

        """self.model = SpeechToTextTranslationModel(
            num_encoder_layers=NUM_ENCODER_LAYERS,
            num_decoder_layers=NUM_DECODER_LAYERS,
            embed_dim=EMBED_DIM,
            num_attn_heads=NUM_HEADS,
            tgt_vocab_size=self.sentence_piece.get_piece_size(),
            d_ff=D_FF,
            dropout=DROPOUT,
            input_feat_dim=AudioParams.n_mels
        )"""

    def train(self):
        training_args = TrainingArguments(
            output_dir=str(TRAINING_OUTPUT_DIR),
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE,
            eval_strategy=EVAL_STRATEGY,
            num_train_epochs=NUM_TRAIN_EPOCHS,
            fp16=FP16,
            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            warmup_steps=WARMUP_STEPS,
            save_steps=SAVE_STEPS,
            eval_steps=EVAL_STEPS,
            logging_steps=LOGGING_STEPS,
            save_total_limit=SAVE_TOTAL_LIMIT,
            load_best_model_at_end=LOAD_BEST_MODEL_AT_END,
            metric_for_best_model=METRIC_FOR_BEST_MODEL,
            greater_is_better=GREATER_IS_BETTER,
            remove_unused_columns=False,
            label_names=["labels"],
            # TPU-specific settings
            dataloader_drop_last=True,  # Required for TPU
            tpu_num_cores=8,  # Adjust based on your TPU
        )

        data_collator = PaddingDataCollator()

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=self.train_dataset,
            eval_dataset=self.eval_dataset,
            compute_metrics=self._metrics,
            data_collator=data_collator,
        )

        # Manually save the custom model config
        # TRAINING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        # with open(TRAINING_OUTPUT_DIR / f"config_{int(time.time())}.json", "w") as f:
        #    json.dump(self.model.config, f)
        TRAINING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        with open(TRAINING_OUTPUT_DIR / f"config_{int(time.time())}.json", "w") as f:
            f.write(self.model.config.to_json_string())

        # Save config using the standard method
        # TRAINING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        # self.model.config.save_pretrained(TRAINING_OUTPUT_DIR)

        print("======= Training =======")
        trainer.train()
        print("======= Training done, saving model =======")
        trainer.save_model()
        print(f"Model saved in: {TRAINING_OUTPUT_DIR}")

    def _metrics(self, pred):
        labels_ids = pred.label_ids
        pred_ids = pred.predictions.argmax(-1)
        labels_ids[labels_ids == -100] = self.sentence_piece.pad_id()
        pred_str = self.sentence_piece.decode(pred_ids.tolist())
        label_str = self.sentence_piece.decode(labels_ids.tolist())
        bleu_metric = evaluate.load("sacrebleu")
        result = bleu_metric.compute(predictions=pred_str, references=[[l] for l in label_str])
        return {"bleu": result["score"]}


In [29]:
# ==================================================
# Inference
# ==================================================


class InferenceEngine:
    def __init__(self, model_checkpoint_path: Path):
        self.device = DEVICE
        self.audio_params = AudioParams()
        self.training_results_dir = TRAINING_OUTPUT_DIR

        # Load CMVN statistics
        cmvn_path = OUTPUT_DIR / "cmvn_train.json"
        with open(cmvn_path, "r") as f:
            cmvn = json.load(f)
        self.mean = torch.tensor(cmvn["mean"], device=self.device).unsqueeze(1)
        self.std = torch.tensor(cmvn["std"], device=self.device).unsqueeze(1)

        # Load SentencePiece model
        spm_path = OUTPUT_DIR / "text_outputs" / "spm" / f"spm_shared_{TextParams.spm_model_type}_{TextParams.spm_vocab_size}.model"
        self.sp = spm.SentencePieceProcessor(model_file=str(spm_path))

        # Load from-scratch model
        model_config = json.load(open(self.training_results_dir / "config.json"))
        self.model = SpeechToTextTranslationModel(**model_config)
        self.model.load_state_dict(load_file(model_checkpoint_path / "model.safetensors", device=self.device))
        self.model.to(self.device)
        self.model.eval()
        print("Inference engine ready.")

    def _extract_features(self, audio_path: str) -> torch.Tensor:
        """Replicates the feature extraction from AudioProcessor."""
        y, sr = librosa.load(audio_path, sr=self.audio_params.sr_target, mono=True)

        n_fft = int(sr * self.audio_params.win_ms / 1000)
        hop_length = int(sr * self.audio_params.hop_ms / 1000)

        S = librosa.feature.melspectrogram(
            y=y, sr=sr, n_mels=self.audio_params.n_mels,
            n_fft=n_fft, hop_length=hop_length
        )
        log_S = np.log(np.maximum(S, 1e-10))

        # Apply CMVN normalization
        features = torch.from_numpy(log_S).to(self.device)
        norm_features = (features - self.mean) / self.std

        # Transpose to (Time, Freq) and add batch dimension
        return norm_features.T.unsqueeze(0)

    def translate_audio(self, audio_path: str, max_len: int = 100) -> str:
        """Translates an audio file using a custom greedy decoding loop."""
        input_features = self._extract_features(audio_path)

        with torch.no_grad():
            src = self.model.feature_projection(input_features) * math.sqrt(self.model.config["embed_dim"])
            src = self.model.pos_encoder(src)
            memory = src
            for layer in self.model.encoder_stack:
                memory = layer(memory, None)

            tgt_tokens = torch.tensor([[self.sp.bos_id()]], dtype=torch.long, device=self.device)

            for _ in range(max_len):
                tgt_mask = self.model._make_pad_lookahead_target_mask(tgt_tokens, self.device)
                tgt_emb = self.model.tgt_embedding(tgt_tokens) * math.sqrt(self.model.config["embed_dim"])
                tgt_emb = self.model.pos_encoder(tgt_emb)
                dec_output = tgt_emb
                for layer in self.model.decoder_stack:
                    dec_output = layer(dec_output, memory, tgt_mask, None)

                logits = self.model.generator(dec_output[:, -1, :])
                next_token = logits.argmax(dim=-1).unsqueeze(0)

                if next_token.item() == self.sp.eos_id():
                    break

                tgt_tokens = torch.cat([tgt_tokens, next_token.T], dim=1)

        return self.sp.decode(tgt_tokens[0].tolist())


"""
elif args.mode == "infer":
    if not args.audio_path:
        raise ValueError("--audio_path is required for inference mode.")

    print("========= Starting Inference Mode =========")
    best_checkpoint_dir = find_best_checkpoint(config.TRAINING_OUTPUT_DIR)
    print(f"Using best checkpoint: {best_checkpoint_dir}")

    engine = InferenceEngine(model_checkpoint_path=best_checkpoint_dir)
    translation = engine.translate_audio(args.audio_path)

    print("\n" + "=" * 50)
    print("TRANSLATION RESULT")
    print("=" * 50)
    print(f"Input Audio: {args.audio_path}")
    print(f"German Translation: {translation}")
    print("=" * 50)
    """

'\nelif args.mode == "infer":\n    if not args.audio_path:\n        raise ValueError("--audio_path is required for inference mode.")\n\n    print("========= Starting Inference Mode =========")\n    best_checkpoint_dir = find_best_checkpoint(config.TRAINING_OUTPUT_DIR)\n    print(f"Using best checkpoint: {best_checkpoint_dir}")\n\n    engine = InferenceEngine(model_checkpoint_path=best_checkpoint_dir)\n    translation = engine.translate_audio(args.audio_path)\n\n    print("\n" + "=" * 50)\n    print("TRANSLATION RESULT")\n    print("=" * 50)\n    print(f"Input Audio: {args.audio_path}")\n    print(f"German Translation: {translation}")\n    print("=" * 50)\n    '

In [30]:
training_manager = TrainingManager()
training_manager.train()

Applying subset: original size 1515298
Using subset fraction: 0.01, resulting in 15152 samples.
Using subset of training data: 15152 samples out of 1515298


comet_ml is installed but the Comet API Key is not configured. Please set the `COMET_API_KEY` environment variable to enable Comet logging. Check out the documentation for other ways of configuring it: https://www.comet.com/docs/v2/guides/experiment-management/configure-sdk/#set-the-api-key


======= Training =======
Loading features from: common_voice_en_19562092.npy
Loading features from: common_voice_en_19794054.npy
Loading features from: common_voice_en_17285689.npy
Loading features from: common_voice_en_18548743.npy
Loading features from: common_voice_en_19801086.npy
Loading features from: common_voice_en_19254748.npy
Loading features from: common_voice_en_18680556.npy
Loading features from: common_voice_en_204513.npy
Loading features from: common_voice_en_18671097.npy
Loading features from: common_voice_en_17585259.npy
Loading features from: common_voice_en_143861.npy
Loading features from: common_voice_en_19769220.npy
Loading features from: common_voice_en_15733742.npy
Loading features from: common_voice_en_24046.npy
Loading features from: common_voice_en_19367677.npy
Loading features from: common_voice_en_19647336.npy
Loading features from: common_voice_en_19251356.npy
Loading features from: common_voice_en_204519.npy
Loading features from: common_voice_en_18636776.

Step,Training Loss,Validation Loss


Loading features from: common_voice_en_526794.npy
Loading features from: common_voice_en_18877722.npy
Loading features from: common_voice_en_19905957.npy
Loading features from: common_voice_en_693222.npy
Loading features from: common_voice_en_596466.npy
Loading features from: common_voice_en_104552.npy
Loading features from: common_voice_en_18814996.npy
Loading features from: common_voice_en_603942.npy
Loading features from: common_voice_en_18844747.npy
Loading features from: common_voice_en_194461.npy
Loading features from: common_voice_en_522805.npy
Loading features from: common_voice_en_19694581.npy
Loading features from: common_voice_en_19683669.npy
Loading features from: common_voice_en_18754826.npy
Loading features from: common_voice_en_19617666.npy
Loading features from: common_voice_en_18994365.npy
Loading features from: common_voice_en_676205.npy
Loading features from: common_voice_en_18796618.npy
Loading features from: common_voice_en_599229.npy
Loading features from: common_


KeyboardInterrupt



In [ ]:
# ========================================================
# Main Script
# ========================================================


def find_best_checkpoint(output_dir: Path) -> Path:
    """Finds the best checkpoint directory saved by the Trainer."""
    checkpoints = [d for d in output_dir.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")]
    if not checkpoints:
        raise FileNotFoundError(f"No checkpoint found in {output_dir}")
    return max(checkpoints, key=lambda d: int(d.name.split('-')[1]))


def main():
    """Main function to run preprocessing, training, or inference based on user input."""

    # Command line argument parser setup
    parser = argparse.ArgumentParser(description="Speech-to-Text Translation Pipeline")
    parser.add_argument(
        "--mode",
        type=str,
        required=True,
        choices=["preprocess", "train", "infer"],
        help="Select mode: 'preprocess' data, 'train' model, or 'infer' with trained model."
    )
    parser.add_argument(
        "--audio_path",
        type=str,
        help="Path to audio file to be translated (required for 'infer' mode)."
    )
    args = parser.parse_args()

    if args.mode == "preprocess":
        print("========= In Preprocessing Mode =========")
        run_preprocessing(
            base_dir=COMMON_VOICE_BASE_DATA_DIR,
            covost_tsv=COVOST_TSV_PATH,
            output_dir=OUTPUT_DIR
        )
        print("========= Preprocessing Mode Done! =========")

    elif args.mode == "train":
        print("========= In Training Mode ========")
        training_manager = TrainingManager()
        training_manager.train()
        print("========= Training Mode Done! =========")



if __name__ == "__main__":
    main()
